# Feature Engineering

## Objective

Transform the raw healthcare appointment data into a clean,
machine-learning-ready dataset.

The feature engineering process includes:

- Data cleaning
- Date and time feature extraction
- Waiting-time calculation
- Age-group creation
- Target preparation
- Removal of identifier columns
- Final dataset validation
- Saving the processed dataset

In [7]:
import numpy as np
import pandas as pd

In [8]:
df = pd.read_csv('../data/raw/HealthCare.csv')

print('Original shape:', df.shape)
df.head()

Original shape: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [9]:
# convert it to numeric for No-show column 

In [10]:
df['No-show']

0         No
1         No
2         No
3         No
4         No
          ..
110522    No
110523    No
110524    No
110525    No
110526    No
Name: No-show, Length: 110527, dtype: object

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  object 
 3   ScheduledDay    110527 non-null  object 
 4   AppointmentDay  110527 non-null  object 
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  object 
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  object 
dtypes: float64(1), int64(8), object(5)
memory usage: 11.8+ MB


In [15]:
df["No_show"] = (
    df["No-show"]
    .astype(str)
    .str.strip()
    .map({
        "No": 0,
        "Yes": 1,
        "0": 0,
        "1": 1
    })
)

In [16]:
print(df["No_show"].dtype)
print(df["No_show"].value_counts(dropna=False))

int64
No_show
0    88208
1    22319
Name: count, dtype: int64


In [17]:
print(df[["No-show", "No_show"]].head(10))

  No-show  No_show
0      No        0
1      No        0
2      No        0
3      No        0
4      No        0
5      No        0
6     Yes        1
7     Yes        1
8      No        0
9      No        0


In [18]:
# clean age

In [19]:
df["Age"] = df["Age"].mask(
    (df["Age"] < 0) | (df["Age"] > 100)
)

print("Missing Age:", df["Age"].isna().sum())

Missing Age: 8


In [20]:
# Convert date columns

In [21]:
df['ScheduledDay'] = pd.to_datetime(
    df['ScheduledDay'],
    utc = True
)

df['AppointmentDay'] = pd.to_datetime(
    df['AppointmentDay'],
    utc = True
)

In [22]:
df[['ScheduledDay', 'AppointmentDay']].head(10)

,ScheduledDay,AppointmentDay
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00
5,2016-04-27 08:36:51+00:00,2016-04-29 00:00:00+00:00
6,2016-04-27 15:05:12+00:00,2016-04-29 00:00:00+00:00
7,2016-04-27 15:39:58+00:00,2016-04-29 00:00:00+00:00
8,2016-04-29 08:02:16+00:00,2016-04-29 00:00:00+00:00
9,2016-04-27 12:48:25+00:00,2016-04-29 00:00:00+00:00


In [23]:
# create waiting days
df['WaitingDays'] = (
    df['AppointmentDay'].dt.normalize()
    - df['ScheduledDay'].dt.normalize()
).dt.days

In [24]:
df['WaitingDays'] = df['WaitingDays'].mask(
    df['WaitingDays'] < 0
)

In [25]:
print('Missing waiting days:', df['WaitingDays'].isna().sum())
print(df['WaitingDays'].describe())

Missing waiting days: 5
count    110522.000000
mean         10.184253
std          15.255115
min           0.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
Name: WaitingDays, dtype: float64


In [26]:
# create schedulerhour

In [27]:
df['ScheduledHour'] = df['ScheduledDay'].dt.hour

In [28]:
df['ScheduledHour'].value_counts().sort_index()

ScheduledHour
6      1578
7     19213
8     15349
9     12823
10    11056
11     8462
12     5422
13     9036
14     9127
15     8079
16     5542
17     2909
18     1340
19      488
20      100
21        3
Name: count, dtype: int64

In [29]:
# Scheduled weekday

In [30]:
df['ScheduledDayOfWeek'] = (
    df['ScheduledDay'].dt.day_name()
)

In [31]:
# Appointment weekday

In [32]:
df['AppointmentDayOfWeek'] = (
    df['AppointmentDay'].dt.day_name()
)

In [33]:
# Appointment month

In [34]:
df['AppointmentMonth'] = (
    df['AppointmentDay'].dt.month
)

In [35]:
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,No_show,WaitingDays,ScheduledHour,ScheduledDayOfWeek,AppointmentDayOfWeek,AppointmentMonth
0,2.987250e+13,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62.0,JARDIM DA PENHA,0,1,0,0,0,0,No,0,0.0,18,Friday,Friday,4
1,5.589978e+14,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56.0,JARDIM DA PENHA,0,0,0,0,0,0,No,0,0.0,16,Friday,Friday,4
2,4.262962e+12,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62.0,MATA DA PRAIA,0,0,0,0,0,0,No,0,0.0,16,Friday,Friday,4
3,8.679512e+11,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8.0,PONTAL DE CAMBURI,0,0,0,0,0,0,No,0,0.0,17,Friday,Friday,4
4,8.841186e+12,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56.0,JARDIM DA PENHA,0,1,1,0,0,0,No,0,0.0,16,Friday,Friday,4


In [36]:
# Create age group

In [37]:
def create_age_group(age):
    if pd.isna(age):
        return 'Unknown'
    elif age <= 12:
        return 'Child'
    elif age <= 19:
        return 'Teen'
    elif age <= 39:
        return 'Young Adult'
    elif age <= 59:
        return 'Adult'
    elif age <= 74:
        return 'Middle age'
    else:
        return '75+'

df['AgeGroup'] = df['Age'].apply(create_age_group)

In [38]:
df['AgeGroup'].value_counts()

AgeGroup
Adult          30072
Young Adult    28870
Child          21036
Middle age     15237
Teen            9375
75+             5929
Unknown            8
Name: count, dtype: int64

In [39]:
# create waiting group

In [40]:
def create_waiting_group(days):
    if pd.isna(days):
        return "Unknown"
    elif days == 0:
        return "Same Day"
    elif days <= 3:
        return "1-3 Days"
    elif days <= 7:
        return "4-7 Days"
    elif days <= 14:
        return "8-14 Days"
    elif days <= 30:
        return "15-30 Days"
    elif days <= 60:
        return "31-60 Days"
    else:
        return "61-180 Days"

df["WaitingGroup"] = df["WaitingDays"].apply(
    create_waiting_group
)

In [41]:
df["WaitingGroup"].value_counts()

WaitingGroup
Same Day       38563
4-7 Days       17510
15-30 Days     17371
1-3 Days       14675
8-14 Days      12025
31-60 Days      8283
61-180 Days     2095
Unknown            5
Name: count, dtype: int64

In [42]:
# create ScheduledTimeGroup

In [43]:
def create_time_group(hour):
    if pd.isna(hour):
        return "Unknown"
    elif hour < 9:
        return "Morning"
    elif hour < 12:
        return "Late Morning"
    elif hour < 15:
        return "Afternoon"
    elif hour < 18:
        return "Evening"
    else:
        return "Late Evening"

df["ScheduledTimeGroup"] = df["ScheduledHour"].apply(
    create_time_group
)

In [44]:
df["ScheduledTimeGroup"].value_counts()

ScheduledTimeGroup
Morning         36140
Late Morning    32341
Afternoon       23585
Evening         16530
Late Evening     1931
Name: count, dtype: int64

In [45]:
print(df.shape)
print(df.columns.tolist())

(110527, 23)
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show', 'No_show', 'WaitingDays', 'ScheduledHour', 'ScheduledDayOfWeek', 'AppointmentDayOfWeek', 'AppointmentMonth', 'AgeGroup', 'WaitingGroup', 'ScheduledTimeGroup']


In [46]:
# create df_processed

In [47]:
columns_to_drop = [
    "PatientId",
    "AppointmentID",
    "ScheduledDay",
    "AppointmentDay",
    "No-show"
]

df_processed = df.drop(
    columns=columns_to_drop
)

In [48]:
print("Processed shape:", df_processed.shape)
print("\nColumns:")
print(df_processed.columns.tolist())

Processed shape: (110527, 18)

Columns:
['Gender', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No_show', 'WaitingDays', 'ScheduledHour', 'ScheduledDayOfWeek', 'AppointmentDayOfWeek', 'AppointmentMonth', 'AgeGroup', 'WaitingGroup', 'ScheduledTimeGroup']


In [49]:
print(df_processed["No_show"].value_counts(dropna=False))

No_show
0    88208
1    22319
Name: count, dtype: int64


In [50]:
print("Target missing values:", df_processed["No_show"].isna().sum())

Target missing values: 0


In [51]:
missing_summary = pd.DataFrame({
    "Missing": df_processed.isnull().sum(),
    "Percentage": (
        df_processed.isnull().sum()
        / len(df_processed)
        * 100
    ).round(2)
})

missing_summary[
    missing_summary["Missing"] > 0
]

,Missing,Percentage
Age,8,0.01
WaitingDays,5,0.00


In [52]:
from pathlib import Path

Path("../data/processed").mkdir(
    parents=True,
    exist_ok=True
)

df_processed.to_csv(
    "../data/processed/healthcare_processed.csv",
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


In [53]:
processed_check = pd.read_csv(
    "../data/processed/healthcare_processed.csv"
)

print("Saved dataset shape:", processed_check.shape)

print("\nTarget distribution:")
print(processed_check["No_show"].value_counts())

print("\nMissing values:")
print(processed_check.isnull().sum())

Saved dataset shape: (110527, 18)

Target distribution:
No_show
0    88208
1    22319
Name: count, dtype: int64

Missing values:
Gender                  0
Age                     8
Neighbourhood           0
Scholarship             0
Hipertension            0
Diabetes                0
Alcoholism              0
Handcap                 0
SMS_received            0
No_show                 0
WaitingDays             5
ScheduledHour           0
ScheduledDayOfWeek      0
AppointmentDayOfWeek    0
AppointmentMonth        0
AgeGroup                0
WaitingGroup            0
ScheduledTimeGroup      0
dtype: int64
